In [1]:
import pandas as pd
df = pd.read_csv("C:\\final year project\\Dataset.csv")
df.head()

,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [2]:
vital_cols = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"
]


In [3]:

window = 10

for col in vital_cols:
    df[f"{col}_mean"] = df[col].rolling(window).mean()
    df[f"{col}_max"]  = df[col].rolling(window).max()
    df[f"{col}_min"]  = df[col].rolling(window).min()
    df[f"{col}_trend"] = df[col] - df[col].shift(window)


In [4]:
df.isna().sum().sort_values(ascending=False).head(10)


Bilirubin_direct    1549220
Fibrinogen          1541968
TroponinI           1537429
Bilirubin_total     1529069
Alkalinephos        1527269
AST                 1527027
Lactate             1510764
PTT                 1506511
SaO2                1498649
EtCO2               1494574
dtype: int64

In [5]:
missing_ratio = df.isna().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.9].index

print("Dropping columns:", len(cols_to_drop))

df = df.drop(columns=cols_to_drop)


Dropping columns: 29


In [6]:
df = df.fillna(method="ffill").fillna(method="bfill").fillna(0)


C:\Users\parid\AppData\Local\Temp\ipykernel_15344\1468440689.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(method="bfill").fillna(0)


In [7]:
df = df.fillna(method="bfill").fillna(method="ffill").fillna(0)


C:\Users\parid\AppData\Local\Temp\ipykernel_15344\3281750956.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="bfill").fillna(method="ffill").fillna(0)


In [8]:
X = df.drop("SepsisLabel", axis=1)
y = df["SepsisLabel"]


In [9]:
from sklearn.preprocessing import StandardScaler

X = df.drop("SepsisLabel", axis=1)
y = df["SepsisLabel"]




In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, stratify = y, random_state = 42)

In [11]:
!pip install catboost
from catboost import CatBoostClassifier
model = CatBoostClassifier(
    iterations = 10000,
    learning_rate = 0.01,
    depth = 6,
    loss_function = 'Logloss',
    eval_metric = 'PRAUC',
    class_weights = [1,20],
    verbose = 100
)

Defaulting to user installation because normal site-packages is not writeable


In [12]:
history = model.fit(X_train, y_train, eval_set = (X_test,y_test) , use_best_model = True)

0:	learn: 0.5772918	test: 0.5652920	best: 0.5652920 (0)	total: 783ms	remaining: 2h 10m 29s
100:	learn: 0.6141356	test: 0.6094437	best: 0.6094766 (99)	total: 1m 24s	remaining: 2h 17m 38s
200:	learn: 0.6252088	test: 0.6208058	best: 0.6208058 (200)	total: 2m 44s	remaining: 2h 13m 39s
300:	learn: 0.6321842	test: 0.6275064	best: 0.6275064 (300)	total: 3m 37s	remaining: 1h 56m 34s
400:	learn: 0.6393328	test: 0.6342406	best: 0.6342406 (400)	total: 5m 5s	remaining: 2h 1m 48s
500:	learn: 0.6458518	test: 0.6398775	best: 0.6398775 (500)	total: 5m 50s	remaining: 1h 50m 38s
600:	learn: 0.6519473	test: 0.6453539	best: 0.6453539 (600)	total: 7m 17s	remaining: 1h 54m 2s
700:	learn: 0.6579852	test: 0.6503645	best: 0.6503645 (700)	total: 8m 20s	remaining: 1h 50m 34s
800:	learn: 0.6637028	test: 0.6552391	best: 0.6552391 (800)	total: 9m 38s	remaining: 1h 50m 45s
900:	learn: 0.6694508	test: 0.6601466	best: 0.6601466 (900)	total: 10m 43s	remaining: 1h 48m 16s
1000:	learn: 0.6746773	test: 0.6645487	best: 0.6

In [13]:
model.save_model('catboost_model.cbm')

In [14]:
from sklearn.metrics import average_precision_score

y_pred_prob = model.predict_proba(X_test)[:, 1]
auprc = average_precision_score(y_test, y_pred_prob)

print("REAL Test AUPRC:", auprc)


REAL Test AUPRC: 0.41479607857157075


In [15]:
from sklearn.metrics import roc_auc_score

y_pred_prob = model.predict_proba(X_test)[:, 1]
auroc = roc_auc_score(y_test, y_pred_prob)

print("Test AUROC:", auroc)

Test AUROC: 0.9569124454787032


In [ ]:
def calculate_lead_times_batch(X_test, y_test, probs, threshold=0.5):
    """
    Calculate lead times for all patients in test set.
    Shows how many hours before sepsis onset the model first predicted it.
    
    Args:
        X_test: test features
        y_test: test labels (1 = sepsis, 0 = no sepsis)
        probs: predicted probabilities for all test samples
        threshold: probability threshold for sepsis alert
    
    Returns:
        list of lead times in hours for patients with sepsis
    """
    lead_times = []
    
    for idx in range(len(X_test)):
        # Get predictions up to this patient
        patient_probs = probs[:idx+1]
        
        # When does model first predict sepsis (exceed threshold)?
        first_alert = None
        for hour, prob in enumerate(patient_probs):
            if prob >= threshold:
                first_alert = hour
                break
        
        # Did this patient actually have sepsis?
        has_sepsis = y_test.iloc[idx] == 1
        actual_hour = idx
        
        # If model predicted and patient has sepsis, calculate lead time
        if first_alert is not None and has_sepsis:
            lead_time = actual_hour - first_alert
            lead_times.append(lead_time)
    
    return lead_times


# Calculate lead times on test set
lead_times = calculate_lead_times_batch(X_test, y_test, y_pred_prob, threshold=0.5)

if lead_times:
    print("\n" + "="*50)
    print("EARLY PREDICTION ANALYSIS")
    print("="*50)
    print(f"\nAverage Lead Time: {np.mean(lead_times):.1f} hours")
    print(f"Median Lead Time: {np.median(lead_times):.1f} hours")
    print(f"Min Lead Time: {min(lead_times):.1f} hours")
    print(f"Max Lead Time: {max(lead_times):.1f} hours")
    print(f"\nPatients with 6+ hour lead time: {sum(1 for lt in lead_times if lt >= 6)}/{len(lead_times)} ({100*sum(1 for lt in lead_times if lt >= 6)/len(lead_times):.1f}%)")
    print(f"Patients with 12+ hour lead time: {sum(1 for lt in lead_times if lt >= 12)}/{len(lead_times)} ({100*sum(1 for lt in lead_times if lt >= 12)/len(lead_times):.1f}%)")
    print(f"Patients with 24+ hour lead time: {sum(1 for lt in lead_times if lt >= 24)}/{len(lead_times)} ({100*sum(1 for lt in lead_times if lt >= 24)/len(lead_times):.1f}%)")
else:
    print("No sepsis cases detected in test set with early prediction")
